
# Weather Dataset — Data Preprocessing

## Objective

Prepare the cleaned and quality-flagged district-level annual NASA POWER weather dataset for exploratory analysis and integration with agricultural data.

## Preprocessing Tasks

- Load and verify the quality-flagged dataset.
- Validate identifiers, years, and data types.
- Separate geography metadata from annual weather measurements.
- Check district-year key uniqueness.
- Inspect weather feature distributions and quality flags.
- Prepare a modeling-ready structure without unjustified imputation or scaling.

## Preprocessing Principles

- Preserve the original source measurements and quality flags.
- Do not interpret missing values as zero.
- Do not remove extreme observations solely because they were flagged.
- Do not fit scalers or imputers before defining the modeling workflow.
- Treat calendar-year weather summaries as annual features, not crop-specific growing-season measurements.

## Input

`data/processed/weather_regional/district_weather_annual_cleaned_flagged_2013_2024.csv`

## Expected Output

A validated, structured weather dataset suitable for downstream analysis, with preprocessing decisions documented and reproducible.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent

input_path = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "district_weather_annual_cleaned_flagged_2013_2024.csv"
)

if not input_path.exists():
    raise FileNotFoundError(f"Cleaned weather dataset not found: {input_path}")

weather_df = pd.read_csv(input_path)

print("Input path:", input_path)
print("Dataset shape:", weather_df.shape)
print("Year range:", weather_df["calendar_year"].min(),
      "to", weather_df["calendar_year"].max())
print("Unique districts:", weather_df["apy_geography_id"].nunique())
print("Unique states/territories:", weather_df["State"].nunique())
print("Missing values:", int(weather_df.isna().sum().sum()))

display(weather_df.head())


Input path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\district_weather_annual_cleaned_flagged_2013_2024.csv
Dataset shape: (7680, 29)
Year range: 2013 to 2024
Unique districts: 640
Unique states/territories: 32
Missing values: 0


,apy_geography_id,State,District,district_latitude,district_longitude,coordinate_source,geography_confidence,nearest_grid_latitude,nearest_grid_longitude,nearest_grid_elevation_m,...,tmax_valid_days,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,precipitation_valid_days,QC_Extreme_Annual_Min_Temperature,QC_Extreme_Annual_Max_Temperature,QC_High_Annual_Precipitation,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance
0,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,365,2456.73,6.730767,29.00,365,False,False,False,False,False
1,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,365,1972.96,5.405370,51.98,365,False,False,False,False,False
2,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,365,1767.49,4.842438,36.97,365,False,False,False,False,False
3,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,366,2187.42,5.976557,52.15,366,False,False,False,False,False
4,APY-GEO-0001,Andaman And Nicobar Islands,Nicobars,6.998715,93.82365,BOUNDARY_REPRESENTATIVE_POINT,PROVISIONAL_BOUNDARY_MATCH,7.0,93.75,9.53,...,365,2155.98,5.906795,48.42,365,False,False,False,False,False


In [2]:

# Cell 3: Validate identifiers, years, and data types

# Check required identifiers
required_columns = [
    "apy_geography_id",
    "State",
    "District",
    "calendar_year"
]

missing_columns = [
    col for col in required_columns
    if col not in weather_df.columns
]

# Validate district-year uniqueness
duplicate_keys = weather_df.duplicated(
    subset=["apy_geography_id", "calendar_year"]
).sum()

# Check expected year coverage
expected_years = set(range(2013, 2025))
actual_years = set(weather_df["calendar_year"].dropna().astype(int))
missing_years = sorted(expected_years - actual_years)
unexpected_years = sorted(actual_years - expected_years)

# Check records per district
records_per_district = weather_df.groupby(
    "apy_geography_id"
)["calendar_year"].nunique()

# Check identifier consistency
district_state_pairs = weather_df[
    ["apy_geography_id", "State"]
].drop_duplicates().groupby("apy_geography_id")["State"].nunique()

district_name_pairs = weather_df[
    ["apy_geography_id", "District"]
].drop_duplicates().groupby("apy_geography_id")["District"].nunique()

print("Missing required columns:", missing_columns)
print("Duplicate district-year keys:", int(duplicate_keys))
print("Missing years:", missing_years)
print("Unexpected years:", unexpected_years)
print("Districts with other than 12 years:",
      int((records_per_district != 12).sum()))
print("Geography IDs linked to multiple states:",
      int((district_state_pairs > 1).sum()))
print("Geography IDs linked to multiple district names:",
      int((district_name_pairs > 1).sum()))

print("\nData types:")
display(weather_df.dtypes.to_frame("Data Type"))


Missing required columns: []
Duplicate district-year keys: 0
Missing years: []
Unexpected years: []
Districts with other than 12 years: 0
Geography IDs linked to multiple states: 0
Geography IDs linked to multiple district names: 0

Data types:


,Data Type
apy_geography_id,object
State,object
District,object
district_latitude,float64
district_longitude,float64
coordinate_source,object
geography_confidence,object
nearest_grid_latitude,float64
nearest_grid_longitude,float64
nearest_grid_elevation_m,float64


In [3]:

# Cell 4: Separate geography metadata and annual weather features

geography_columns = [
    "apy_geography_id",
    "State",
    "District",
    "district_latitude",
    "district_longitude",
    "coordinate_source",
    "geography_confidence",
    "nearest_grid_latitude",
    "nearest_grid_longitude",
    "nearest_grid_elevation_m",
    "nearest_grid_distance_km",
    "latitude",
    "longitude"
]

key_columns = [
    "apy_geography_id",
    "calendar_year"
]

weather_feature_columns = [
    "tmin_mean_c",
    "tmin_min_c",
    "tmax_mean_c",
    "tmax_max_c",
    "precipitation_total_mm",
    "precipitation_mean_mm_day",
    "precipitation_max_mm_day"
]

valid_day_columns = [
    "tmin_valid_days",
    "tmax_valid_days",
    "precipitation_valid_days"
]

quality_flag_columns = [
    col for col in weather_df.columns
    if col.startswith("QC_")
]

# Create separate analytical views; preserve the original DataFrame
geography_df = weather_df[geography_columns].drop_duplicates().copy()

weather_features_df = weather_df[
    key_columns
    + weather_feature_columns
    + valid_day_columns
    + quality_flag_columns
].copy()

print("Original dataset shape:", weather_df.shape)
print("Unique geography metadata shape:", geography_df.shape)
print("Annual weather features shape:", weather_features_df.shape)
print("Weather measurement features:", len(weather_feature_columns))
print("Valid-day fields:", len(valid_day_columns))
print("Quality flags:", quality_flag_columns)

print("\nDuplicate geography IDs in metadata:",
      int(geography_df["apy_geography_id"].duplicated().sum()))

print("Duplicate district-year keys in weather features:",
      int(weather_features_df.duplicated(key_columns).sum()))

print("\nWeather feature preview:")
display(weather_features_df.head())


Original dataset shape: (7680, 29)
Unique geography metadata shape: (640, 13)
Annual weather features shape: (7680, 17)
Weather measurement features: 7
Valid-day fields: 3
Quality flags: ['QC_Extreme_Annual_Min_Temperature', 'QC_Extreme_Annual_Max_Temperature', 'QC_High_Annual_Precipitation', 'QC_Extreme_Daily_Precipitation', 'QC_Long_District_Grid_Distance']

Duplicate geography IDs in metadata: 0
Duplicate district-year keys in weather features: 0

Weather feature preview:


,apy_geography_id,calendar_year,tmin_mean_c,tmin_min_c,tmax_mean_c,tmax_max_c,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,tmin_valid_days,tmax_valid_days,precipitation_valid_days,QC_Extreme_Annual_Min_Temperature,QC_Extreme_Annual_Max_Temperature,QC_High_Annual_Precipitation,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance
0,APY-GEO-0001,2013,27.472904,25.78,28.279671,30.11,2456.73,6.730767,29.00,365,365,365,False,False,False,False,False
1,APY-GEO-0001,2014,27.517479,25.74,28.430301,30.75,1972.96,5.405370,51.98,365,365,365,False,False,False,False,False
2,APY-GEO-0001,2015,27.664959,26.03,28.533370,30.15,1767.49,4.842438,36.97,365,365,365,False,False,False,False,False
3,APY-GEO-0001,2016,27.918333,25.43,28.810492,31.98,2187.42,5.976557,52.15,366,366,366,False,False,False,False,False
4,APY-GEO-0001,2017,27.570219,25.54,28.360055,29.62,2155.98,5.906795,48.42,365,365,365,False,False,False,False,False


In [4]:

# Cell 5: Summarize weather features and quality flags

print("WEATHER FEATURE SUMMARY")
display(
    weather_features_df[weather_feature_columns]
    .describe()
    .T
    .round(3)
)

print("\nQUALITY FLAG SUMMARY")
flag_summary = pd.DataFrame({
    "Flag": quality_flag_columns,
    "Flagged_Records": [
        int(weather_features_df[col].sum())
        for col in quality_flag_columns
    ],
    "Flagged_Districts": [
        int(
            weather_features_df.loc[
                weather_features_df[col], "apy_geography_id"
            ].nunique()
        )
        for col in quality_flag_columns
    ]
})

display(flag_summary)

print("\nVALID-DAY COUNTS")
for col in valid_day_columns:
    print(f"\n{col}")
    display(
        weather_features_df[col]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("Records")
    )

print("\nRecords with at least one quality flag:",
      int(weather_features_df[quality_flag_columns].any(axis=1).sum()))


WEATHER FEATURE SUMMARY


,count,mean,std,min,25%,50%,75%,max
tmin_mean_c,7680.0,18.904,4.342,-11.804,18.646,19.664,20.572,28.328
tmin_min_c,7680.0,5.677,6.491,-35.460,3.060,5.190,8.660,26.440
tmax_mean_c,7680.0,30.413,4.811,-0.887,29.973,31.853,32.971,36.642
tmax_max_c,7680.0,42.020,6.190,14.190,39.540,44.100,46.320,49.840
precipitation_total_mm,7680.0,1295.309,681.737,136.260,857.770,1141.545,1522.840,6693.070
precipitation_mean_mm_day,7680.0,3.546,1.866,0.373,2.349,3.127,4.172,18.337
precipitation_max_mm_day,7680.0,72.922,37.144,13.340,47.300,64.715,88.272,404.510



QUALITY FLAG SUMMARY


,Flag,Flagged_Records,Flagged_Districts
0,QC_Extreme_Annual_Min_Temperature,11,3
1,QC_Extreme_Annual_Max_Temperature,498,184
2,QC_High_Annual_Precipitation,13,9
3,QC_Extreme_Daily_Precipitation,11,9
4,QC_Long_District_Grid_Distance,3396,283



VALID-DAY COUNTS

tmin_valid_days


,Records
tmin_valid_days,
365,5760
366,1920



tmax_valid_days


,Records
tmax_valid_days,
365,5760
366,1920



precipitation_valid_days


,Records
precipitation_valid_days,
365,5760
366,1920



Records with at least one quality flag: 3718


In [5]:

# Cell 6: Validate feature values and prepare preprocessing report

validation_checks = {
    "Negative precipitation total": (
        weather_features_df["precipitation_total_mm"] < 0
    ),
    "Negative precipitation mean": (
        weather_features_df["precipitation_mean_mm_day"] < 0
    ),
    "Negative precipitation maximum": (
        weather_features_df["precipitation_max_mm_day"] < 0
    ),
    "Mean minimum temperature exceeds mean maximum": (
        weather_features_df["tmin_mean_c"]
        > weather_features_df["tmax_mean_c"]
    ),
    "Annual minimum exceeds annual maximum": (
        weather_features_df["tmin_min_c"]
        > weather_features_df["tmax_max_c"]
    ),
    "Precipitation total below mean daily precipitation": (
        weather_features_df["precipitation_total_mm"]
        < weather_features_df["precipitation_mean_mm_day"]
    ),
    "Mean daily precipitation exceeds daily maximum": (
        weather_features_df["precipitation_mean_mm_day"]
        > weather_features_df["precipitation_max_mm_day"]
    ),
}

validation_report = pd.DataFrame([
    {
        "Check": check_name,
        "Violations": int(condition.sum()),
        "Status": "PASS" if not condition.any() else "REVIEW"
    }
    for check_name, condition in validation_checks.items()
])

display(validation_report)

print("Any missing weather measurements:",
      bool(weather_features_df[weather_feature_columns].isna().any().any()))

print("Any missing quality flags:",
      bool(weather_features_df[quality_flag_columns].isna().any().any()))

print("Original dataset still has", weather_df.shape[0],
      "rows and", weather_df.shape[1], "columns.")


,Check,Violations,Status
0,Negative precipitation total,0,PASS
1,Negative precipitation mean,0,PASS
2,Negative precipitation maximum,0,PASS
3,Mean minimum temperature exceeds mean maximum,0,PASS
4,Annual minimum exceeds annual maximum,0,PASS
5,Precipitation total below mean daily precipita...,0,PASS
6,Mean daily precipitation exceeds daily maximum,0,PASS


Any missing weather measurements: False
Any missing quality flags: False
Original dataset still has 7680 rows and 29 columns.


In [6]:

# Cell 7: Create and verify the modeling-ready weather table

model_weather_df = weather_features_df.copy()

# Ensure year and key fields have appropriate types
model_weather_df["calendar_year"] = model_weather_df[
    "calendar_year"
].astype("int64")

# Define feature groups for downstream workflows
model_feature_columns = weather_feature_columns.copy()
model_key_columns = ["apy_geography_id", "calendar_year"]
model_quality_columns = quality_flag_columns.copy()
model_valid_day_columns = valid_day_columns.copy()

# Verify the table structure
assert model_weather_df[model_key_columns].notna().all().all()
assert not model_weather_df.duplicated(model_key_columns).any()
assert model_weather_df[model_feature_columns].notna().all().all()
assert model_weather_df.shape[0] == weather_df.shape[0]

print("Modeling-ready table shape:", model_weather_df.shape)
print("Unique districts:",
      model_weather_df["apy_geography_id"].nunique())
print("Year range:",
      model_weather_df["calendar_year"].min(),
      "to",
      model_weather_df["calendar_year"].max())
print("Weather feature columns:", model_feature_columns)
print("Valid-day columns:", model_valid_day_columns)
print("Quality flag columns:", model_quality_columns)
print("Duplicate district-year keys:",
      int(model_weather_df.duplicated(model_key_columns).sum()))
print("Missing values:", int(model_weather_df.isna().sum().sum()))

display(model_weather_df.head())


Modeling-ready table shape: (7680, 17)
Unique districts: 640
Year range: 2013 to 2024
Weather feature columns: ['tmin_mean_c', 'tmin_min_c', 'tmax_mean_c', 'tmax_max_c', 'precipitation_total_mm', 'precipitation_mean_mm_day', 'precipitation_max_mm_day']
Valid-day columns: ['tmin_valid_days', 'tmax_valid_days', 'precipitation_valid_days']
Quality flag columns: ['QC_Extreme_Annual_Min_Temperature', 'QC_Extreme_Annual_Max_Temperature', 'QC_High_Annual_Precipitation', 'QC_Extreme_Daily_Precipitation', 'QC_Long_District_Grid_Distance']
Duplicate district-year keys: 0
Missing values: 0


,apy_geography_id,calendar_year,tmin_mean_c,tmin_min_c,tmax_mean_c,tmax_max_c,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,tmin_valid_days,tmax_valid_days,precipitation_valid_days,QC_Extreme_Annual_Min_Temperature,QC_Extreme_Annual_Max_Temperature,QC_High_Annual_Precipitation,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance
0,APY-GEO-0001,2013,27.472904,25.78,28.279671,30.11,2456.73,6.730767,29.00,365,365,365,False,False,False,False,False
1,APY-GEO-0001,2014,27.517479,25.74,28.430301,30.75,1972.96,5.405370,51.98,365,365,365,False,False,False,False,False
2,APY-GEO-0001,2015,27.664959,26.03,28.533370,30.15,1767.49,4.842438,36.97,365,365,365,False,False,False,False,False
3,APY-GEO-0001,2016,27.918333,25.43,28.810492,31.98,2187.42,5.976557,52.15,366,366,366,False,False,False,False,False
4,APY-GEO-0001,2017,27.570219,25.54,28.360055,29.62,2155.98,5.906795,48.42,365,365,365,False,False,False,False,False


In [7]:

# Cell 8: Save the preprocessed weather dataset

preprocessed_path = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "district_weather_annual_preprocessed_2013_2024.csv"
)

preprocessed_path.parent.mkdir(parents=True, exist_ok=True)

model_weather_df.to_csv(preprocessed_path, index=False)

# Reload and verify the saved file
saved_weather_df = pd.read_csv(preprocessed_path)

print("Saved path:", preprocessed_path)
print("File exists:", preprocessed_path.exists())
print("Saved dataset shape:", saved_weather_df.shape)
print("Saved missing values:", int(saved_weather_df.isna().sum().sum()))
print(
    "Saved duplicate district-year keys:",
    int(saved_weather_df.duplicated(
        ["apy_geography_id", "calendar_year"]
    ).sum())
)
print(
    "Saved column names match:",
    list(saved_weather_df.columns) == list(model_weather_df.columns)
)
print(
    "Saved data matches in-memory table:",
    saved_weather_df.equals(model_weather_df)
)


Saved path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\district_weather_annual_preprocessed_2013_2024.csv
File exists: True
Saved dataset shape: (7680, 17)
Saved missing values: 0
Saved duplicate district-year keys: 0
Saved column names match: True
Saved data matches in-memory table: True
